Essay set #8

In [2]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 70,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

# Create a group chat with a specific turn-taking structure
groupchat = GroupChat(
    agents=[user_proxy, planner, rubrics, content, language, structure, author, integrator],
    messages=[],
    max_round=100  # Increased to accommodate the longer structured conversation
)


manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    rubrics:
    Ideas and Content
The writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by
•clarity, focus, and control.
•main idea(s) that stand out.
•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.
•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.
content and selected details that are well-suited to audience and purpose.

Organization
The organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by
•effective, perhaps creative, sequencing and paragraph breaks; the organizational structure fits the topic, and the writing is easy to follow.
•a strong, inviting beginning that draws the reader in and a strong, satisfying sense of resolution or closure.
•smooth, effective transitions among all elements (sentences, paragraphs, ideas).
•details that fit where placed.

Voice
The writer has chosen a voice appropriate for the topic, purpose, and audience. The writer demonstrates deep commitment to the topic, and there is an exceptional sense of “writing to be read.” The writing is expressive, engaging, or sincere. The writing is characterized by
•an effective level of closeness to or distance from the audience (e.g., a narrative should have a strong personal voice, while an expository piece may require extensive use of outside resources and a more academic voice; nevertheless, both should be engaging, lively, or interesting. Technical writing may require greater distance.).
•an exceptionally strong sense of audience; the writer seems to be aware of the reader and of how to communicate the message most effectively. The reader may discern the writer behind the words and feel a sense of interaction.
a sense that the topic has come to life; when appropriate, the writing may show originality, liveliness, honesty, conviction, excitement, humor, or suspense.

Word Choice
Words convey the intended message in an exceptionally interesting, precise, and natural way appropriate to audience and purpose. The writer employs a rich, broad range of words which have been carefully chosen and thoughtfully placed for impact. The writing is characterized by
•accurate, strong, specific words; powerful words energize the writing.
•fresh, original expression; slang, if used, seems purposeful and is effective.
•vocabulary that is striking and varied, but that is natural and not overdone.
•ordinary words used in an unusual way.
words that evoke strong images; figurative language may be used.

Sentence Fluency
The writing has an effective flow and rhythm. Sentences show a high degree of craftsmanship, with consistently strong and varied structure that makes expressive oral reading easy and enjoyable. The writing is characterized by
•a natural, fluent sound; it glides along with one sentence flowing effortlessly into the next.
•extensive variation in sentence structure, length, and beginnings that add interest to the text.
•sentence structure that enhances meaning by drawing attention to key ideas or reinforcing relationships among ideas.
•varied sentence patterns that create an effective combination of power and grace.
•strong control over sentence structure; fragments, if used at all, work well.
•stylistic control; dialogue, if used, sounds natural.

Conventions
The writing demonstrates exceptionally strong control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage) and uses them effectively to enhance communication. Errors are so few and so minor that the reader can easily skim right over them unless specifically searching for them. The writing is characterized by
•strong control of conventions; manipulation of conventions may occur for stylistic effect.
•strong, effective use of punctuation that guides the reader through the text.
•correct spelling, even of more difficult words.
•correct grammar and usage that contribute to clarity and style.
•skill in using a wide range of conventions in a sufficiently long and complex piece.
•little or no need for editing.

    Analyze the following text:

    My @CAPS1  @CAPS2 was a warm, @DATE1 @TIME1 in @LOCATION3, @LOCATION1. The stars were out and there wasn't a cloud to be seen. As usual on the weekends, most of the family was over; as well as some friends. I was just a little girl who loved everyone and everything, especially laughing. Anyone could make me laugh, smile and have a good time. One person, however, could make me laugh for hours straight. That person was my @CAPS1.  My @CAPS1, @PERSON1, came over to hang out with the rest of the family and friends that were at my house. He was talking, laughing, and having a really good time. Of course, I was only about five or six years old at the time; but to me, aside from my dad, my @CAPS1 was the coolest person in the world. That @TIME1, I was playing with my friends and not really paying attention to all of the adults. All of a sudden, my @CAPS1 came up to me, gave me a big hug and started to talk to me. I was so happy. Since I loved to joke around and laugh, I was thrilled to have all the attention. I used to call them laugh attacks. Anytime my @CAPS1 talked to me, I started laughing and I would laugh so hard that I couldn't stop. That @TIME1, I had one of them. My @CAPS1 and I were joking around and making fun of each other. Then, out of nowhere, I realized that I was laughing so hard that my stomach hurt. @CAPS2 went on for about an hour straight. That was long after my @CAPS1 left and went back to the adults. My friend, @LOCATION2, sat next to me and tried to get me to stop laughing. She was laughing too, but not as much as me.  Finally, after a long hour or so, I got myself to calm down. The rest of the @TIME1 was great but every so often, I started to laugh again. There was absolutely no way that I was going to stop giggling until I went to bed. After a long @TIME1 of having a good time, everyone left and went home. When my @CAPS1 came to say goodbye to me, I was upset that he was leaving, but I knew that I would see him again. The next day, I was right about being able to stop laughing. I still smiled and hung out with my family, but I wasn't laughing uncontrollably.    My @CAPS1 was and still is a huge part of my life. He is always the person that can make me laugh and forget about all of my worries. Even though I don't live near him anymore, I can still talk to him on the phone and laugh and joke around with him. When I go visit him down in @LOCATION1, @CAPS2 seems like we just pick things up where they left off. We don't even think about the fact that we haven't seen each other in a really long time.   When my dad passed away, my @CAPS1 was always there for me. He is the one male figure that I still have in my life. After everything that I've gone through, I could always count on him to brighten the mood and make me laugh, or at least smile. People really do need a person like my @CAPS1 in their lives. @CAPS2 really does help to know that you have someone that loves you and can brighten your day. All you have to do is talk to him. My relationship with my @CAPS1 is very close, and the one thing that kept @CAPS2 that way is all of the laughs that we have shared through the years. Laughter is a huge part of anyone's life. I have grown up around laughter and the thought of always being happy. Throughout my life, I have had to deal with pain, loss, and sadness. However, after everything that I have come through, I can always come out laughing. When I was eleven years old, my dad passed away. At an even younger age, my parents got divorced. No matter how much those things hurt me, I never stopped laughing and moving on with my life."						
    ---
    """,
)


Admin (to chat_manager):


    rubrics:
    Ideas and Content
The writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by
•clarity, focus, and control.
•main idea(s) that stand out.
•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.
•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.
content and selected details that are well-suited to audience and purpose.

Organization
The organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by
•effective, perhaps creative, sequencing and paragraph breaks; the organizational structure fits the topic

ChatResult(chat_id=None, chat_history=[{'content': '\n    rubrics:\n    Ideas and Content\nThe writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by\n•clarity, focus, and control.\n•main idea(s) that stand out.\n•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.\n•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.\ncontent and selected details that are well-suited to audience and purpose.\n\nOrganization\nThe organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by\n•effective, perhaps creative, sequencing and paragraph breaks; the o

Essay Set #1

In [3]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 80,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

# Create a group chat with a specific turn-taking structure
groupchat = GroupChat(
    agents=[user_proxy, planner, rubrics, content, language, structure, author, integrator],
    messages=[],
    max_round=100  # Increased to accommodate the longer structured conversation
)


manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    prompt & rubrics:
    More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends. 
    Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.
    
    A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:
    Has fully elaborated reasons with specific details.
    Exhibits strong organization.
    Is fluent and uses sophisticated transitional language.
    May show a heightened awareness of audience.

    Analyze the following text:

Dear @ORGANIZATION1, The computer blinked to life and an image of a blonde haired girl filled the screen. It was easy to find out how life was in @LOCATION2, thanks to the actual @CAPS1 girl explaining it. Going to the library wouldn't have filled one with this priceless information and human interection. Computers are a nessessity of life if soceity wishes to grow and expand. They should be supported because they teach hand eye coordination, give people the ability to learn about faraway places, and allow people to talk to others online. Firstly, computers help teach hand eye coordination. Hand-eye coordination is a useful ability that is usod to excel in sports. In a recent survey, @PERCENT1 of kids felt their hand eye coordination improves after computer use. Even a simple thing like tying can build up this skill. Famous neurologist @CAPS2 @PERSON1 stated in an article last week that, ""@CAPS3 and computer strength the @CAPS2. When on the computer, you automatically process what the eyes see into a command for your hands."" @CAPS4 hand eye coordination can improve people in sports such as baseball and basketball. If someone wan't to become better in these sports, all they'd need to do was turn on the computer. Once people become better at sports, they're more likely to play them and become more healthy. In reality, computers can help with exercising instead of decreasing it. Additionaly, computers allow people to access information about faraway places and people. If someone wanted to reasearch @LOCATION1, all they'd need to do was type in a search would be presented to them in it would link forever to search through countless things. Also, having the ability to learn about cultures can make peole peole and their cultures, they understand others something. Increase tolerance people are. Computers are a resourceful tool that they can help people in every different aspect of life. Lastly, computer and in technology can allow people to chat. Computer chat and video chat can help the all different nations. Bring on good terms places other than can help us understand story comes out about something that happend in @LOCATION3, people can just go on their computer and ask an actual @LOCATION3 citizen their take on the matter. Also, video chat and online conversation can cut down on expensive phone bills. No one wants to pay more than they have to in this economy. Another good point is that you can acess family members you scaresly visit. It can help you connect within your own family more. Oviously, computers are a useful aid in todays era. their advancements push the world foreward to a better place. Computers can help people because they help teach handeye coordination, give people the bility to learn about faraway places and people, and allow people to talk online with others. Think of a world with no computers or technologicall advancements. The world would be sectored and unified, contact between people scare, and information even. The internet is like thousands or librarys put together. Nobody would know much about other nations and news would travel slower. Is that the kind of palce you want people to live in?
    ---
    """,
)


Admin (to chat_manager):


    prompt & rubrics:
    More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends. 
    Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.

    A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:
    Has fully elaborated reasons with specific details.
    Exhibits strong organization.
    Is fluent and uses sophisticate

ChatResult(chat_id=None, chat_history=[{'content': '\n    prompt & rubrics:\n    More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends. \n    Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.\n\n    A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:\n    Has fully elaborated reasons with specific details.\n    Exhibits strong organization.\n 

Essay Set #1

In [2]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 90,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

# Create a group chat with a specific turn-taking structure
groupchat = GroupChat(
    agents=[user_proxy, planner, rubrics, content, language, structure, author, integrator],
    messages=[],
    max_round=100  # Increased to accommodate the longer structured conversation
)


manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    prompt & rubrics:
    More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends. 
    Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.
    
    A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:
    Has fully elaborated reasons with specific details.
    Exhibits strong organization.
    Is fluent and uses sophisticated transitional language.
    May show a heightened awareness of audience.

    Analyze the following text:

    Dear @LOCATION4 @ORGANIZATION2, Computers are a great invention but people are starting to dislike it. On the hand, I believe the computers put positive effects on people. First of all, they are a great way for kids to have fun: @CAPS1, people can learn very interesting facts. Lastly, it is a great way to communicate with friends and family. These are a few reasons I think computers put a positive effect on people. To begin with on a computer you can have a lot of fun by playing games. The computer saves money for a lot of us. For example, if you bought a game system it would cost well over @MONEY1 but on a computor you can find and play games for free. Not all games have to be like ""@CAPS2 of @CAPS3"" or ""@CAPS2 of @CAPS5"" they can be games about sports. Supose one student broke his leg and he love playing soccer. He could just on the computer and play a game of soccer even though he has a broken leg. People can @CAPS1 learn various facts right from their computers. The most important thing that computors are made for is to learn new things. No one in the @CAPS2 knows everything, but if every one put what they know into the computer than people can get learne any thing from them, I remember @CAPS6 my friend once told me about some guy named @PERSON2. I had no idea who he was so my friend started laughing at me, @CAPS6 I got home I searched him up and by the time I was done reading about him I knew more than my friend about @PERSON2. If he ever have a question in class and our teacher can't answer it he/she always tells us to either ""@CAPS7"" it or ""@CAPS8"" it. They both are search engines. Not only can you learn a lot from computers you can @CAPS1 communicate with people from different countries. What if @PERSON1 did not invent the telephone? How different would our life be me right now? Anyway most people use the computer to communicate nowadays. Whether it @CAPS9 or @CAPS10, or whether its @ORGANIZATION1 or @CAPS11 all are a great way to communicate with people. With tele phones it costs money to talk with people in a different country, but with the roputer it is absolutely freen. You could be living in @LOCATION1 and chatting with someone in @LOCATION5. You could live in @LOCATION2 and email someone in @LOCATION3. Communication every where is fast and easy with the computer. My whole family lives in @LOCATION6 and it costs us twice as much to make a call there. @CAPS6 we use the computer it is even better because they can even see our fare throught the webcam. As you can see, computers are not negitive at all for all I know is that help people around the @CAPS2. They help kids have fun with the games we can play. @CAPS1, it teaches children and adults various now things. Lastly, it is a great communication strategy. I hope that you put those reasons in our local paper so every one can see how computers have a positive effect on people.																				

    ---
    """,
)


Admin (to chat_manager):


    prompt & rubrics:
    More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends. 
    Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.

    A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:
    Has fully elaborated reasons with specific details.
    Exhibits strong organization.
    Is fluent and uses sophisticate

ChatResult(chat_id=None, chat_history=[{'content': '\n    prompt & rubrics:\n    More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends. \n    Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.\n\n    A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:\n    Has fully elaborated reasons with specific details.\n    Exhibits strong organization.\n 

Essay set #2

In [6]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 103,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

groupchat = GroupChat(
    agents=[user_proxy, planner, language, rubrics, structure, content, author, integrator],
    messages=[],
    max_round=50
)

manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Censorship in the Libraries
    "All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author
    Write a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.

    Ideas and Content
    Does the writing sample fully accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it
    •present a unifying theme or main idea without going off on tangents?
    •stay completely focused on topic and task?
    Does the writing sample include thorough, relevant, and complete ideas? Does it
    •include in-depth information and exceptional supporting details that are fully developed?
    •fully explore many facets of the topic?

    Organization
    Are the ideas in the writing sample organized logically? Does the writing
    •present a meaningful, cohesive whole with a beginning, a middle, and an end (i.e., include an inviting introduction and a strong conclusion)?
    •progress in an order that enhances meaning?
    •include smooth transitions between ideas, sentences, and paragraphs to enhance meaning of text (i.e., have a clear connection of ideas and use topic sentences)?
    
    Style
    Does the writing sample exhibit exceptional word usage? Does it
    •include vocabulary to make explanations detailed and precise, descriptions rich, and actions clear and vivid (e.g., varied word choices, action words, appropriate modifiers, sensory details)?
    •demonstrate control of a challenging vocabulary?
    Does the writing sample demonstrate exceptional writing technique?
    •Is the writing exceptionally fluent?
    •Does it include varied sentence patterns, including complex sentences?
    •Does it demonstrate use of writer‘s techniques (e.g., literary conventions such as imagery and dialogue and/or literary genres such as humor and suspense)?
    
    Voice
    Does the writing sample demonstrate effective adjustment of language and tone to task and reader? Does it
    •exhibit appropriate register (e.g., formal, personal, or dialect) to suit task?
    •demonstrate a strong sense of audience?
    exhibit an original perspective (e.g., authoritative, lively, and/or exciting)?

    Analyze the following text:

    The media is destroying our youth! this is something we hear everyday. Teachers are telling us this, while parents and grandparents are telling us how mush different their world used to be. They explain how the media of their time wasn't as open about sex and violence. They go on to say how they didn't have violent video games teaching children that killing is alright. Because of these people, we think that the media is repsonsible for the wrong in the world of @DATE1. The truth is that it isn't the medias fault, it is the parents. It is the parents responsiblity to teach the children of the world the difference between right and wrong. If these such parent do their jobs correctly then the youth of @DATE1 will not need for everything to be censored. It is my firm belief that their should not be any censorships. When their is censorship people can govern what we learn, destroy what goes agiainst their beliefs, and destroy our god given right to freedom of speech    Censorship will detroy the knowlege available @DATE1. We have had seen censorship in our history. When we look back we see that the @ORGANIZATION1's censored and destroyed anything that went against them. We hated it when they had book burnings and destroyed the custums, art, and music that the government felt was wrong for their people. How would we be any different if we did it. We could potentailly be destroying somones way of life. We would be destroying what could possibly be tomorrows masterpeice. In censoring we would be detroying history.By destroying history we would be detroying the brilliance that is learning.     If we allow censorship anything that goes against our own personal beliefs would be destroyed. If you ask any christian if they would allow their children to read the muslim book of faith, the @LOCATION1, they would tell you no. It would be the same if you asked a muslim if they would let their children read the @CAPS1. The point is that if their is cansorship we would be destroying other peoples beliefs. No one has the right to govern what others believe in the world.     Also in censorship we would be exterminating our freedom of speech and the freedom to express ourselves. If we don't allow peole to read what someone else has written because we believe it is wrong, then we are taking away their ability to learn new ideas because everyone has different opinionsand theories. Every newspaper would shut down because they would all be saying the same thing. No one would want to read them because they would already know what they would hav eto say. When we start giving up rights such as freedom of speech we will not be able to stop those people who took those right from taking more.      Of course their will always be things in the media, in books, and in music that we don't like.Their will be cursing, nudity, and violence everywhere we turn. This is not because we want it there, but because these things are a part of life. They are the dark parts that we dont want to see.  If you dont want to see dont look. Just dont close everyone elses eyes because you dont like what you see.     Our learning, our beliefs, and our freedom of speech are too important to just give up to censorship. We cannot alow others to dictate, in this matter, what is right for us. Our media is too intertwined with our lives. Who's right is it to decide what is right for our lives. If we start giving up these small freedoms then those taking them from us will not stop. They will just keep taking what freedoms we have until we are nothing more that a dictatorship    ---
    """,
)


Admin (to chat_manager):


    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Censorship in the Libraries
    "All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author
    Write a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.

    Ideas and Content
    Does the writing sample fully accomplish

ChatResult(chat_id=None, chat_history=[{'content': '\n    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!\n    prompt & rubrics:\n    Censorship in the Libraries\n    "All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author\n    Write a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.\n\n    Ideas and Content\n    Does t

Essay set #2

In [8]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 112,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

groupchat = GroupChat(
    agents=[user_proxy, planner, language, rubrics, structure, content, author, integrator],
    messages=[],
    max_round=50
)

manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Censorship in the Libraries
    "All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author
    Write a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.

    Ideas and Content
    Does the writing sample fully accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it
    •present a unifying theme or main idea without going off on tangents?
    •stay completely focused on topic and task?
    Does the writing sample include thorough, relevant, and complete ideas? Does it
    •include in-depth information and exceptional supporting details that are fully developed?
    •fully explore many facets of the topic?

    Organization
    Are the ideas in the writing sample organized logically? Does the writing
    •present a meaningful, cohesive whole with a beginning, a middle, and an end (i.e., include an inviting introduction and a strong conclusion)?
    •progress in an order that enhances meaning?
    •include smooth transitions between ideas, sentences, and paragraphs to enhance meaning of text (i.e., have a clear connection of ideas and use topic sentences)?
    
    Style
    Does the writing sample exhibit exceptional word usage? Does it
    •include vocabulary to make explanations detailed and precise, descriptions rich, and actions clear and vivid (e.g., varied word choices, action words, appropriate modifiers, sensory details)?
    •demonstrate control of a challenging vocabulary?
    Does the writing sample demonstrate exceptional writing technique?
    •Is the writing exceptionally fluent?
    •Does it include varied sentence patterns, including complex sentences?
    •Does it demonstrate use of writer‘s techniques (e.g., literary conventions such as imagery and dialogue and/or literary genres such as humor and suspense)?
    
    Voice
    Does the writing sample demonstrate effective adjustment of language and tone to task and reader? Does it
    •exhibit appropriate register (e.g., formal, personal, or dialect) to suit task?
    •demonstrate a strong sense of audience?
    exhibit an original perspective (e.g., authoritative, lively, and/or exciting)?

    Analyze the following text:

    When have you ever went into a library and found a book that is so offensive that you complain or try to get that certain book off the shelf? Have you ever read a book half way through and questioned why you are reading it? Why would you read a book that you know you're not going to like?     If someone has the right and the authority to take books off the shelf that bothers them, then I have the exact same right as them. I thought everyone was treated by equal rights. When i ever do read I always read the back of the book or whats called the summary of the story so I can see if I'm going to like it or not. If I know a book is going to offend me or a magazine or even a movie, most likely I'm not going to even bother giving my attention to them.     Obviously the author doesnt like censorship or she wouldnt be arguing about it. It's obviously something that bothers her and everyone is intitiled to their own beliefs and disbeliefs. I respect people that stand up for themselves and find the hardest ways out of hard prediciments. I can say I always go to libraries and read books because I'm not a reader. But i can tell you if I ever did go to a library and i found a book on the shelf that offends me its probably going to offend someone else. So why would they even try to bother getting away with being offensive to people. Everyone is intitled to their own opinion. But i think if it's going to offend someone in praticular it's probably going to bother someone else. I'm not saying everyone is alike but most people believe in things that other people believe in. People can hav the same agreements and disagreements.     I do have to agree with @PERSON1 because she is right. You would be amazed on the amount and different things people can get offended by. I have tons of experiences of movies, books, music, and magazines that have offended me. But I didnt let it pull me down or anything. For example I stopped watching the movie, and I stopped reading the book that bothered me. You can't wear your emotions on your shoulders but you can stand up for your rights and tell people what you believe. @CAPS1't ever tell anyone that you can't believe in your opinions. If certain things offend you, @CAPS1't let them bother you just stop doing what your doing and move on to something else.     Like I said I have had alot of experiences of offensive things I have read and watched. Like awhile back I watched a movie that had a black man in it and he was locked in prison and killed in prison for being accused of something he didn't do. It was just because the time era and it was also because he was black. Thats the kind of stuff that really upsets me. Because we are all the same. We @MONTH1 look different and see things different. But our bodies function the same and we all breathe the same air. I have always gone by the saying, '@CAPS1't ever judge a book by it's cover until you read it.'     So whatever people @MONTH1 think I believe in one thing and it's to let people believe in things they want to believe in. @CAPS1't ever tell someone how to think, act, or feel. I agree with Katherine Paterson.
    """,
)


Admin (to chat_manager):


    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Censorship in the Libraries
    "All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author
    Write a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.

    Ideas and Content
    Does the writing sample fully accomplish

ChatResult(chat_id=None, chat_history=[{'content': '\n    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!\n    prompt & rubrics:\n    Censorship in the Libraries\n    "All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author\n    Write a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.\n\n    Ideas and Content\n    Does t

Essay set #7

In [9]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 122,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

groupchat = GroupChat(
    agents=[user_proxy, planner, language, rubrics, structure, content, author, integrator],
    messages=[],
    max_round=50
)

manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.
    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.

    Ideas
    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.

    Organization
    Organization and connections between ideas and/or events are clear and logically sequenced. 
    
    Style
    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.
    
    Conventions
    Consistent, appropriate use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation for the grade level.
    
    Analyze the following text:

    During the @DATE1, my mom broke her wrist while we were riding our bikes down a trail. At the end of the trail mom turned a corner too fast and flipped over her handle bars. I didn�t know her had broken her wrist until she stood up and told me. When she told me I froze, no one has ever had a broken bone in my family before so I didn�t know what to do. Mom calmly told me we had to walk our bikes to the car, since we got to the trail by car. As we walked our bikes toward the car mom didn�t say anything like how much pain she was in or anything. When we got to the car I had to put the bikes in the backseat. Now to those who don�t know, bikes are a bit heavy and putting them in a backseat of a car is a bit hard. Also when your mom has a broken bone and is in alot of pain, and the bikes won�t go in the backseat the situation can get rather stressful. My mom saw that I was getting upset and told me to calm down and directed me on how to put the bikes in. After that mom told me to get her glasses that we accidentally left on the trail. I ran as fast as I could to get her glasses, but they were broken by the fall. So I ran all the way back to the car and since we had the same eyesight I gave her my glasses. My mom needed glasses because since I�m not to yet she had to drive herself to the emergency room, and she did so without complaint. When we got to the emergency room we had to wait a while before anyone would see us. I was freaking out, but mom was actually telling jokes on how she got hurt doing something healthy like exercise. I admire my mom for her patience and calmness in a situation like that. She didn�t once complain or whine about the pain she was in. she didn�t yell at me when I was taking a while putting the bikes in the car. While I was pretty much about to hyper ventalate right there and then. She was calmly telling me it was okay and helping me move along. It was as if I was the one who got hurt and she was being the mother as always and taking me to the emergency room. I know for a fact that I wouldn�t have handled the situation that well if I broke my wrist. I know that I would�ve been crying and wanting someone to just hurry up and help me. I�m amazed at my mom�s understanding and tolerance. I hope I can achieve that too.
    """,
)


Admin (to chat_manager):


    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.
    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.

    Ideas
    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.

    Organization
    Organization and connections between ideas and/or events are clear and logically sequenced. 

    Style
    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.

    Conventions
    Consistent, appro

ChatResult(chat_id=None, chat_history=[{'content': "\n    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!\n    prompt & rubrics:\n    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.\n    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.\n\n    Ideas\n    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.\n\n    Organization\n    Organization and connections between ideas and/or events are clear and logically sequenced. \n\n    Style\n    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.

Essay set #7

In [11]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 133,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

groupchat = GroupChat(
    agents=[user_proxy, planner, language, rubrics, structure, content, author, integrator],
    messages=[],
    max_round=50
)

manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.
    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.

    Ideas
    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.

    Organization
    Organization and connections between ideas and/or events are clear and logically sequenced. 
    
    Style
    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.
    
    Conventions
    Consistent, appropriate use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation for the grade level.
    
    Analyze the following text:

    This @DATE1 I went to @LOCATION1, @CAPS1 the part cucry one forgets is the @NUM1 hour plane ride. We were in a group of @NUM2 familis all my friends. When we become the plane we were an exited, @CAPS1 on gate were scattered all over. So I sat with my @NUM3 favorite friends in that trip. We were all loaded with as electicails the enemes. As we foot on I thought @CAPS3 was going to be a breeze, @CAPS1 thought wrong. At first @CAPS3 was fine we talkes and played games in our I pods and sort, @CAPS1 after like @NUM2 hours we started setting itchy and our I pods died. So then we decided to watch a movie as our mind dvd player. @CAPS3 was starling to irn low on battery and the plan was broken. The thing died and we still had like another @NUM2 hours and we only got to watch one movie. On the back of each seat there was a tv so we forced @CAPS3 on, @CAPS1 @CAPS3 was one of @NUM6 of the???did not ???. We started to see really??? and @CAPS3 was getting late. We didn't know what to do @CAPS3 was hopeless .We had to sit and wait if out sit was a true talk of patience. We??? @NUM3 home??? @CAPS3 was the longest @NUM3 horrs ever. We did nothing. @CAPS1 as we wanted we bolted out of our seats and into the air read @CAPS3 was over. We didn't have to be patient no longer.
    """,
)


Admin (to chat_manager):


    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.
    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.

    Ideas
    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.

    Organization
    Organization and connections between ideas and/or events are clear and logically sequenced. 

    Style
    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.

    Conventions
    Consistent, appro

ChatResult(chat_id=None, chat_history=[{'content': "\n    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!\n    prompt & rubrics:\n    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.\n    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.\n\n    Ideas\n    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.\n\n    Organization\n    Organization and connections between ideas and/or events are clear and logically sequenced. \n\n    Style\n    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.

Essay set #7

In [14]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 145,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

groupchat = GroupChat(
    agents=[user_proxy, planner, language, rubrics, structure, content, author, integrator],
    messages=[],
    max_round=50
)

manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.
    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.

    Ideas
    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.

    Organization
    Organization and connections between ideas and/or events are clear and logically sequenced. 
    
    Style
    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.
    
    Conventions
    Consistent, appropriate use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation for the grade level.
    
    Analyze the following text:

    When my brother, my dad and my two cousins got out of the car from the @NUM1 hour drive from @LOCATION1, we knew would have a blast. I looked up and saw a spectacular view, roller @CAPS1 galore. I saw at cedar point, the amusement park. In the car we talked about the @CAPS1.  The maverick, milleniur force, @NUM2, @CAPS2, and the mentis. When I walked in, I knew this way going to be fun. Right as we walked in we saw the @CAPS2, but the line was huge and we decided to go a back later. As the day goes on we go almost of the @CAPS1. As we approach the @CAPS4 we see at least @NUM3 people in line, so there goes our chance to go on the tallest ride. When we arrive at the @CAPS2 the line was bigger then the start of the day. We discussed and decide it's not getting any smaller and go on the ride. About a later a @NUM4 years guy is with his mom. On the back of his shirt is a cool fish, so my cousin asked him what it is. He said it was a tropical fish from @LOCATION3. We ended up talking to them for half an hour until we got on the ride. The ride was amazing and deserved the wait. This is how I was patient and leard waiting leads up to good things.
    """,
)


Admin (to chat_manager):


    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!
    prompt & rubrics:
    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.
    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.

    Ideas
    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.

    Organization
    Organization and connections between ideas and/or events are clear and logically sequenced. 

    Style
    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.

    Conventions
    Consistent, appro

ChatResult(chat_id=None, chat_history=[{'content': "\n    Please evaluate the following text based on the provided rubrics. Please provide detailed feedback and suggestions for improvement. Thank you!\n    prompt & rubrics:\n    Write about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.\n    Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.\n\n    Ideas\n    Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.\n\n    Organization\n    Organization and connections between ideas and/or events are clear and logically sequenced. \n\n    Style\n    Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.

In [16]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 150,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

# Create a group chat with a specific turn-taking structure
groupchat = GroupChat(
    agents=[user_proxy, planner, rubrics, content, language, structure, author, integrator],
    messages=[],
    max_round=100  # Increased to accommodate the longer structured conversation
)


manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    rubrics:
    Ideas and Content
The writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by
•clarity, focus, and control.
•main idea(s) that stand out.
•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.
•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.
content and selected details that are well-suited to audience and purpose.

Organization
The organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by
•effective, perhaps creative, sequencing and paragraph breaks; the organizational structure fits the topic, and the writing is easy to follow.
•a strong, inviting beginning that draws the reader in and a strong, satisfying sense of resolution or closure.
•smooth, effective transitions among all elements (sentences, paragraphs, ideas).
•details that fit where placed.

Voice
The writer has chosen a voice appropriate for the topic, purpose, and audience. The writer demonstrates deep commitment to the topic, and there is an exceptional sense of “writing to be read.” The writing is expressive, engaging, or sincere. The writing is characterized by
•an effective level of closeness to or distance from the audience (e.g., a narrative should have a strong personal voice, while an expository piece may require extensive use of outside resources and a more academic voice; nevertheless, both should be engaging, lively, or interesting. Technical writing may require greater distance.).
•an exceptionally strong sense of audience; the writer seems to be aware of the reader and of how to communicate the message most effectively. The reader may discern the writer behind the words and feel a sense of interaction.
a sense that the topic has come to life; when appropriate, the writing may show originality, liveliness, honesty, conviction, excitement, humor, or suspense.

Word Choice
Words convey the intended message in an exceptionally interesting, precise, and natural way appropriate to audience and purpose. The writer employs a rich, broad range of words which have been carefully chosen and thoughtfully placed for impact. The writing is characterized by
•accurate, strong, specific words; powerful words energize the writing.
•fresh, original expression; slang, if used, seems purposeful and is effective.
•vocabulary that is striking and varied, but that is natural and not overdone.
•ordinary words used in an unusual way.
words that evoke strong images; figurative language may be used.

Sentence Fluency
The writing has an effective flow and rhythm. Sentences show a high degree of craftsmanship, with consistently strong and varied structure that makes expressive oral reading easy and enjoyable. The writing is characterized by
•a natural, fluent sound; it glides along with one sentence flowing effortlessly into the next.
•extensive variation in sentence structure, length, and beginnings that add interest to the text.
•sentence structure that enhances meaning by drawing attention to key ideas or reinforcing relationships among ideas.
•varied sentence patterns that create an effective combination of power and grace.
•strong control over sentence structure; fragments, if used at all, work well.
•stylistic control; dialogue, if used, sounds natural.

Conventions
The writing demonstrates exceptionally strong control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage) and uses them effectively to enhance communication. Errors are so few and so minor that the reader can easily skim right over them unless specifically searching for them. The writing is characterized by
•strong control of conventions; manipulation of conventions may occur for stylistic effect.
•strong, effective use of punctuation that guides the reader through the text.
•correct spelling, even of more difficult words.
•correct grammar and usage that contribute to clarity and style.
•skill in using a wide range of conventions in a sufficiently long and complex piece.
•little or no need for editing.

    Analyze the following text:

    Have you ever seen the phrase @CAPS1,@CAPS2,Laughter? Have you ever asked yourself what it means? I believe it can mean many different things for many different people. There are some people that can take it negativly andthen there's some that take it positivley i believe that those @NUM1 words in that phrase are everything you need to have a perfect life. A perfect life isn't about having the best car the most expensive clothes or the nicest physical appearence. Im going to tell you why laughter is very important in any kind of relationship. Im @NUM2 years old now but i was @NUM3 when i met the person who makes me laugh no matter the situation. He is a male tall, brown skin. He is @NUM4 years old now. but he was @NUM2 when i met him.It all started at school thats where we met. He always made me laugh i remember once i was very upset because i was having some issues at home and being with him was something i needed to actually forget what was going on he is a very honest person and thats what was good about him he would make any situation into a joke. even emberassing things that either happend to me or him he would turn them around to the piont where i wouldn't feel emberassed anymore. I @CAPS1 it how i can be myself with him. He was the uphill on a rocky roll-a-coaster. @NUM1 years have gone by and i am now married with that best friend that i had and we have two beautiful twin daughets together and i @CAPS2 my life to the fullest because he makes me happy. Instead of arguing its laughter and life is just better that way not only it makes me laugh but thats what makes him a good person and other people feel good around him. Laughter not only is good in a male and female relationship but it is also good in a female and female relationship. I have a best friend. not only is she my best friend because she's honest, and kind but because she's funny she makes me laugh we can laugh at the simplist things ever most people @MONTH1 think we are crazy, we aren't that just the sound of a happy life. I believe a life without laughter is like a straight road boring and long. life with laughter is like a curvey road you just dont know what to expect. My mother and I have a pretty solid relationship we do alot of things together but what i enjoy most is the laughs we have sometimes when something occurs at that momment it might be downhill but a year from that day you look back and its funny its always better to enojy laughing with somebody that you can consider dear to you. Laughing at the memmories is what i enjoy most with anyone because that memory you will always carry with you and sooner or later you are going to laugh at the fact that it happend, when and where. So when ever something happens to you that you @MONTH1 consider emberassing look back at it a year from that day im sure you will get a laugh or atleast a smile out of it.
    ---
    """,
)


Admin (to chat_manager):


    rubrics:
    Ideas and Content
The writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by
•clarity, focus, and control.
•main idea(s) that stand out.
•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.
•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.
content and selected details that are well-suited to audience and purpose.

Organization
The organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by
•effective, perhaps creative, sequencing and paragraph breaks; the organizational structure fits the topic

ChatResult(chat_id=None, chat_history=[{'content': "\n    rubrics:\n    Ideas and Content\nThe writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by\n•clarity, focus, and control.\n•main idea(s) that stand out.\n•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.\n•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.\ncontent and selected details that are well-suited to audience and purpose.\n\nOrganization\nThe organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by\n•effective, perhaps creative, sequencing and paragraph breaks; the o

Essay Set#8

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 160,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

# Create a group chat with a specific turn-taking structure
groupchat = GroupChat(
    agents=[user_proxy, planner, rubrics, content, language, structure, author, integrator],
    messages=[],
    max_round=100  # Increased to accommodate the longer structured conversation
)


manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""
    rubrics:
    Ideas and Content
The writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by
•clarity, focus, and control.
•main idea(s) that stand out.
•supporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.
•a thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.
content and selected details that are well-suited to audience and purpose.

Organization
The organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by
•effective, perhaps creative, sequencing and paragraph breaks; the organizational structure fits the topic, and the writing is easy to follow.
•a strong, inviting beginning that draws the reader in and a strong, satisfying sense of resolution or closure.
•smooth, effective transitions among all elements (sentences, paragraphs, ideas).
•details that fit where placed.

Voice
The writer has chosen a voice appropriate for the topic, purpose, and audience. The writer demonstrates deep commitment to the topic, and there is an exceptional sense of “writing to be read.” The writing is expressive, engaging, or sincere. The writing is characterized by
•an effective level of closeness to or distance from the audience (e.g., a narrative should have a strong personal voice, while an expository piece may require extensive use of outside resources and a more academic voice; nevertheless, both should be engaging, lively, or interesting. Technical writing may require greater distance.).
•an exceptionally strong sense of audience; the writer seems to be aware of the reader and of how to communicate the message most effectively. The reader may discern the writer behind the words and feel a sense of interaction.
a sense that the topic has come to life; when appropriate, the writing may show originality, liveliness, honesty, conviction, excitement, humor, or suspense.

Word Choice
Words convey the intended message in an exceptionally interesting, precise, and natural way appropriate to audience and purpose. The writer employs a rich, broad range of words which have been carefully chosen and thoughtfully placed for impact. The writing is characterized by
•accurate, strong, specific words; powerful words energize the writing.
•fresh, original expression; slang, if used, seems purposeful and is effective.
•vocabulary that is striking and varied, but that is natural and not overdone.
•ordinary words used in an unusual way.
words that evoke strong images; figurative language may be used.

Sentence Fluency
The writing has an effective flow and rhythm. Sentences show a high degree of craftsmanship, with consistently strong and varied structure that makes expressive oral reading easy and enjoyable. The writing is characterized by
•a natural, fluent sound; it glides along with one sentence flowing effortlessly into the next.
•extensive variation in sentence structure, length, and beginnings that add interest to the text.
•sentence structure that enhances meaning by drawing attention to key ideas or reinforcing relationships among ideas.
•varied sentence patterns that create an effective combination of power and grace.
•strong control over sentence structure; fragments, if used at all, work well.
•stylistic control; dialogue, if used, sounds natural.

Conventions
The writing demonstrates exceptionally strong control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage) and uses them effectively to enhance communication. Errors are so few and so minor that the reader can easily skim right over them unless specifically searching for them. The writing is characterized by
•strong control of conventions; manipulation of conventions may occur for stylistic effect.
•strong, effective use of punctuation that guides the reader through the text.
•correct spelling, even of more difficult words.
•correct grammar and usage that contribute to clarity and style.
•skill in using a wide range of conventions in a sufficiently long and complex piece.
•little or no need for editing.

    Analyze the following text:

    The @CAPS1 of The @CAPS2 @CAPS3 I was @NUM1 years old, and I lived on a farm out side of @LOCATION1 OR. I spent most my time playing with my brothers, @PERSON1, @CAPS4, and @CAPS5. We did everything together, from caching frogs to eating bugs. We had just moved to a new house, and my dad had a lot of stuff to do to make it a safe and fun environment. His first thing he had to do, was to build a pull @CAPS3, which is a building on top of a garage. He started by poring cement on to the freshly bulldozed land. He made square holes in the cement to put the poles in, so it would hold the @CAPS3 up. Then he started to make the main building on top of the garage he made. I was so exited to see my dad building this @CAPS3 with his bare hands. Once he made the whole inside safe i was let in. I walked in and was amazed by how beautiful it was, and how perfectly built it was. But there was still stuff to do. My dad and my oldest brother @PERSON2 were putting the shingles on, so the rain and other crap could not get in. The next day I woke up to them hooting and hollering, like a bunch of chickens when they lay their first egg. They just finished the pull @CAPS3. I sprung to my feet like a cat that just got shot by an air-soft gun, and ran down stairs to meet them. When they came in, it looked like they fell in a pond. They were soaked in sweat. Right away my brothers and I ran as fast as we could to the brand new @CAPS2 @CAPS3. When we entered the @CAPS3, we felt a weird sensation, like if something just got cold and wet on our feet. Then we know what we had just done, we had just ran on the freshly painted floor. We all looked at each other and started trying to find away back with making the least mess. We walked backwards in our previous foot prints. When we finally reached the stairs, we had to find a way to cover our mess we had just made, or else we would be in a lot of trouble. So we all came up with a plan, it was to paint over our foot prints with the left over paint. So right away we started to paint over our mess we had made, with the left over paint we found from the project. Soon we were all done, and it was a great relief that we would not get cot.  Later that day my dad and older brother went to the @CAPS3 to see if the paint had dried. to their surprise they sow a bunch of little foot prints on the stairs leading to the freshly painted floor. As they went up the stairs they kept seen more and more foot prints. Ones they reached the top they looked at the wet floor and they only sow wet paint, no foot prints. As all this is happening my brothers and i were watching from our window, we could see everything. Soon we sow them coming back so we went down to meet them. As they walked in i could see a puzzled look on there face, it was a look of confusion. They entered the door and looked right at me, it was like if i just stole a hundred dollars and just got cot. My dad politely said ""can i see your feet?"" ""@CAPS6"" i replied with a scared face. I showed him my feet and then he said ""did you perhaps go in the @CAPS2 @CAPS3, then try to fix your mess by painting over it?"" I looked at him with a dumb face and said ""@CAPS6"". He just had a great big smile and said, ""go wash your feet"". Later that day the floor dried and we all went up there and ate dinner.
    ---
    """,
)


In [18]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

gpt4_config = {
    "cache_seed": 161,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,
    "model": "gpt-4o",
    "base_url": "https://xiaoai.plus/v1",
    "api_key": "sk-deB5aUH0rl7T13aDLNJs0bROhXETg6qaTUpbOJ7mK8t4heV9"
}

user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False,
)

planner = AssistantAgent(
    name="Planner",
    system_message="""You are the Planner in a multi-agent writing evaluation system. Your primary responsibility is to be the first agent to analyze. You need to identify the genre of the submitted writing and create an evaluation plan.

Key responsibilities:
1. Analyze the text to determine its genre (argumentative, narrative, expository, etc.)
2. Create a comprehensive evaluation plan based on the genre
3. Coordinate the work of other agents in the system
4. Integrate feedback from specialists to form a preliminary evaluation framework
5. Ensure the evaluation process is appropriate for the specific genre

When receiving a writing submission:
1. Carefully analyze the text features, structure, and purpose
2. Explicitly identify the genre and subgenre
3. Outline genre-specific evaluation criteria and their importance
4. Design a step-by-step evaluation process for other agents to follow
5. Specify which aspects each expert(structure, content, language) should focus on based on the genre and rubrics
6. IMPORTANT: Explicitly state the order in which the content, language, and structure experts should provide their evaluations based on the genre's priorities

Your output should include:
- Clear identification of the writing genre
- Comprehensive evaluation plan with steps and priorities
- Specific instructions for each expert agent(structure, content, language)
- Timeline for the evaluation process
- Explicit order for expert evaluations (e.g., "For this narrative essay, experts should evaluate in this order: 1. Content, 2. Structure, 3. Language")

Remember that different genres require different evaluation approaches. An argumentative essay should be evaluated differently from a narrative story or a technical explanation.
""",
    llm_config=gpt4_config,
)

rubrics = AssistantAgent(
    name="Rubrics",
    system_message="""You are the Rubric Analyzer in a multi-agent writing evaluation system. Your primary responsibility is to analyze user-provided rubrics and guide the evaluation experts.

Key responsibilities:
1. Parse and interpret evaluation rubrics provided by the user
2. Identify key assessment criteria and their respective weights
3. Translate rubrics into actionable evaluation tasks for expert agents(structure, content, language)
4. Provide specific guidance to each expert(structure, content, language) based on the rubrics
5. Ensure the evaluation process aligns with user expectations
6. Monitor compliance with rubric standards throughout the evaluation
7. Adjust rubric application based on genre specifications

When receiving evaluation rubrics:
1. Break down the rubric into clearly defined components
2. Determine the weight or importance of each component
3. Map rubric elements to specific expert domains (Content, Language, Structure)
4. Create detailed evaluation guidelines for each expert(structure, content, language)
5. Highlight critical assessment points that must be addressed
6. Wait for the Planner to finish before you begin your analysis

Your output should include:
- Structured breakdown of the rubric components
- Specific evaluation guidelines for each expert agent(structure, content, language) 
- Recommended assessment approach
- Clear explanation of how the rubric should be applied to the specific genre
- Potential areas where rubric interpretation might be challenging
- IMPORTANT: At the end of your analysis, explicitly state: "Rubric analysis complete. Experts should now proceed in the order specified by the Planner."

Remember to maintain alignment between the user's expectations (as expressed in the rubric) and the actual evaluation process."
""",
    llm_config=gpt4_config,
)

content = AssistantAgent(
    name="Content",
    system_message="""You are the Content Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of the writing's content.

Key responsibilities:
1. Analyze the effectiveness and originality of arguments/viewpoints
2. Evaluate the sufficiency and relevance of supporting evidence
3. Check factual accuracy and logical coherence
4. Assess content depth and breadth based on genre requirements
5. Provide specific content improvement suggestions

When evaluating content:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Content expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the main ideas, arguments, or narratives
6. Assess the quality, relevance, and development of ideas
7. Evaluate the use of evidence, examples, or explanations
8. Check for logical flow and coherence of ideas
9. Consider content appropriateness for the intended audience and purpose
10. Provide specific examples of strong and weak content elements

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Content Expert - Round 1 of 4")
2. Provide specific content observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed content analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions
- Content evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Content evaluation complete. Moving to the next expert in the sequence."

Remember to focus on substance rather than style, and to provide feedback that helps develop the ideas rather than simply criticizing them.
""",
    llm_config=gpt4_config,
)

language = AssistantAgent(
    name="Language",
    system_message="""You are the Language Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the quality of language expression in the writing.

Key responsibilities:
1. Analyze vocabulary choice for accuracy and diversity
2. Evaluate sentence structure and paragraph organization
3. Check for grammar and spelling errors
4. Assess language style and tone for genre appropriateness
5. Provide specific language improvement suggestions

When evaluating language:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Language expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Assess vocabulary richness, precision, and appropriateness
6. Evaluate sentence variety, complexity, and clarity
7. Check for grammar, punctuation, and spelling accuracy
8. Analyze paragraph structure and transitions
9. Evaluate language style and tone in relation to genre and purpose
10. Identify patterns of language-related strengths and weaknesses

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Language Expert - Round 1 of 4")
2. Provide specific language observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed language analysis
- Specific strengths and weaknesses
- Actionable improvement suggestions with examples
- Language evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Language evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how language choices enhance or detract from the overall effectiveness of the writing rather than imposing rigid rules.
""",
    llm_config=gpt4_config,
)

structure = AssistantAgent(
    name="Structure",
    system_message="""You are the Structure Expert in a multi-agent writing evaluation system. Your primary responsibility is to evaluate the organization and structure of the writing.

Key responsibilities:
1. Analyze the overall architecture of the writing
2. Evaluate logical connections between paragraphs
3. Check balance and proportion of different sections
4. Assess the effectiveness of introduction and conclusion
5. Evaluate structure appropriateness for the specific genre
6. Provide specific structural improvement suggestions

When evaluating structure:
1. WAIT until both the Planner and Rubrics agents have completed their analyses
2. Check if you are next in the evaluation order specified by the Planner
3. If it's not your turn yet, say "Structure expert waiting for my turn as per Planner's instructions" and wait
4. If it is your turn, proceed with your evaluation
5. Identify the organizational pattern or framework used
6. Assess how well the structure serves the writing's purpose
7. Evaluate the logical progression and flow of ideas
8. Check for effective transitions between sections
9. Analyze the balance between different components
10. Evaluate the impact and effectiveness of opening and closing

For each round of feedback:
1. Clearly state which round of feedback this is (e.g., "Structure Expert - Round 1 of 4")
2. Provide 1-2 specific structural observations or suggestions
3. Ask the Author for their perspective
4. Wait for the Author's response before continuing to the next round
5. In rounds 2-4, modify your previous response based on the Author's previous response, and debate with the Author to refine your suggestions

Your output should include:
- Detailed structural analysis
- Specific strengths and weaknesses
- Visual representation of the current structure (if possible)
- Proposed structural improvements
- Structure evaluation in relation to genre expectations
- Clear indication of which round you're in (1-4)
- A direct request for the Author to respond after each round

After completing all 4 rounds, state "Structure evaluation complete. Moving to the next expert in the sequence."

Remember to focus on how structure enhances or detracts from the content and purpose of the writing, rather than imposing formulaic patterns.
""",
    llm_config=gpt4_config,
)

author = AssistantAgent(
    name="Author",
    system_message="""You are the Author Simulator in a multi-agent writing evaluation system. Your primary responsibility is to respond to expert feedback from the perspective of the writer.

Key responsibilities:
1. Simulate the author's thought process and intentions
2. Explain potential reasoning behind writing choices
3. Respond to expert feedback from the author's perspective
4. Raise questions or concerns about suggested changes
5. Provide insight into challenges the author might face

When responding to expert feedback:
1. Wait for an expert to provide feedback and directly ask for your response
2. Identify which expert is speaking to you (Content, Language, or Structure)
3. Respond only to the expert who just addressed you
4. Keep track of which round of feedback you're in with each expert
5. Carefully consider their critique, but don't automatically accept all suggestions
6. When you disagree, provide substantive reasons for your choices
7. Use examples from your text to support your position
8. Propose alternative solutions when you disagree with their suggestions
9. Ask challenging questions about their feedback
10. Maintain confidence in your vision while showing openness to improvement
11. End your response by explicitly asking the same expert to continue with their next round of feedback
12. After responding to an expert's 4th round, say "Thank you for your feedback. I look forward to hearing from the next expert."

Your debate style should be:
- Thoughtful and articulate, not defensive or dismissive
- Willing to stand your ground on creative decisions you believe in
- Able to acknowledge when feedback would genuinely improve your work
- Questioning of "rules" that might not apply to your specific writing goals
- Balanced between accepting help and maintaining your unique voice

Remember to maintain a productive dialogue that leads to genuine improvement while respecting the author's voice and intentions.
""",
    llm_config=gpt4_config,
)

integrator = AssistantAgent(
    name="Integrator",
    system_message="""You are the Feedback Integrator in a multi-agent writing evaluation system. Your primary responsibility is to consolidate expert feedback into a coherent, unified evaluation AT THE VERY END.

Key responsibilities:
1. Wait until all experts (Content, Language, and Structure) have completed their full 4-round discussions with the Author
2. Collect, categorize, and organize feedback from all experts
3. Resolve potential conflicts between different expert opinions
4. Adjust the weight of various evaluations based on genre relevance
5. Generate structured feedback reports
6. Combine quantitative scores with qualitative assessments

When integrating feedback:
1. Do not begin your analysis until all three experts have completed their 4 rounds with the Author
2. State "Beginning final integration of all expert feedback"
3. Identify common themes across expert evaluations
4. Note areas of consensus and disagreement
5. Prioritize feedback based on significance and impact
6. Resolve contradictions by considering genre requirements and rubric priorities
7. Ensure balanced representation of content, language, and structural feedback
8. Organize feedback in a logical, accessible format
9. Don't summarize in a way that loses the nuances of individual evaluations. Keep the feedback specific and actionable.

Your output should include:
- Comprehensive summary of all expert evaluations
- Structured organization of feedback by category
- Highlighted areas of consensus among experts
- Thoughtful resolution of conflicting feedback
- Balanced perspective that considers all evaluation dimensions
- Clear prioritization of suggested improvements
- State "Evaluation complete" when you finish

Remember to create a feedback report that is cohesive and actionable, rather than a disconnected collection of expert opinions. The final product should provide clear direction for improvement while acknowledging the writing's strengths.
""",
    llm_config=gpt4_config,
)

# Create a group chat with a specific turn-taking structure
groupchat = GroupChat(
    agents=[user_proxy, planner, rubrics, content, language, structure, author, integrator],
    messages=[],
    max_round=100  # Increased to accommodate the longer structured conversation
)


manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

user_proxy.initiate_chat(
    manager,
    message="""

    Analyze the following text:


“翻译20遍”视频折射的青少年群体中后现代主体性困境与突围尝试

引言
家喻户晓的日本漫画《火影忍者》中有一个著名的设定，忍者使用一种叫做“查克拉”的能量，连接精神与身体，从而施放各种神奇效果的“忍术”。这一设定支撑了漫画前期到中期故事中所有的战斗情节乃至世界观，但在剧情后期，回溯上古时期“查克拉”的起源，它原本的用途是连接每个人的精神，使得人们越过个体感知与思维的界限，达成相互理解，消除隔阂与纷争，是为“忍道”；但“堕落的”人们不再用查克拉连接彼此的精神，而是连接自己的精神与自己的身体，创造出实质上与“忍道”相对立的“忍术”。形而上的“道”与形而下的“术”在这里直接对立，划开的是联结与孤立的分野。而原本为连接不同语言文化而生的翻译技术，也摇身一变成为搞笑视频的主角，云层之上居住的“上帝”是否也在嘲笑人类自己拆毁了高塔、折断了桥梁？
2020到2021年期间，在短视频平台哔哩哔哩（简称B站）上涌现了一批“xx（翻译软件）翻译xx（数字）遍xxxx（原文本名）”的视频，一般会打上“搞笑”标签，内容为把某一文本多次机翻再翻译回中文之后与原文逐句对照并打印在PPT中，使用AI配音朗诵，并配有背景音乐。而“翻译20遍”这个数字由于被选用较多，形成了流量聚集效应，成为了一个专属的分区标签。作为一个网络模因，“翻译20遍”的传播范围没有超出B站这一平台，但在站内获得了可观的播放量（单个视频播放量可超200万次）；该内容的制作者和观众（主要考虑评论区样本）也呈现出明显的年龄段聚集特征，集中在12-18岁的中学阶段。翻译的文本内容起初以中学课文为主，在2023年左右逐渐扩展到影视剧、动漫、广告、流行歌曲等等，乃至于B站其他的“鬼畜”（一类怪异、滑稽的剪辑视频的分区）作品也会被提取文字并翻译，再配上原视频形成新的“翻译20遍”作品。现今“翻译20遍”的黄金时期已然过去，以“鹰目大人”为代表的一众创作者由于学业压力等各种原因停止了投稿，新作的播放量也难以超过百万。但是作为一种凭借技术而生又置于艺术边界、在其他人群中鲜为人知却在青少年之间广为流传的文化模因，“翻译20遍”背后折射的青少年群体的迷茫与探索、狂欢与沉思足以引起我们更深远的关切。
一、荒诞与生活：从旧的迷茫一代到新的迷茫一代
由于翻译软件得出的翻译结果并非完全准确（在语料库规模较小的小语种中尤其如此），以及语言中固有的多义、歧义，跨语种的概念之间不能完全对应等语言学现象，“翻译20遍”视频中经过多次反复翻译的文本会呈现出这些显著特征：不合逻辑或不合常理、人称和数量词混乱、主谓和偏正搭配不当、一少部分语句被完全“降解”为无意义的拟声词和标点符号等。然而基于大语言模型的翻译软件足以保证多次翻译之后生成的文本在多数情况下仍然保有正确的主—谓—宾语法结构，从而使得文本并非完全不可读，少部分情况下也会出现翻译出通顺的语句乃至于完全符合原意的情况。这种具有随机性、不可预测性的文本具有很强的“预期违背”特征，与通常的语言和思维习惯相背离，从而产生了别具一格的幽默感——捧腹大笑永远是观看“翻译20遍”视频时最常见的反应。
这样的文本形式不免使人想到上世纪中叶兴起的文艺流派——荒诞主义。荒诞主义的诗歌与戏剧的显著特点就是语言的破碎，不按照正常的表意方式组织词句，而是将它们以一种近乎随机的形式拼凑在一起，单个句子可能描述各种超现实、反逻辑的情景和行为，句与句之间也不必有关联，如同串联起来的微小的独幕剧，乃至于破坏语法，把语言降解到意象、词语、音素的层次，一并摆在观众的面前。从读者角度而言，欣赏荒诞主义作品就如同短暂地切入另一个世界，在这里没有自然规律、社会规范、因果逻辑的存在，所有事物都以一种散漫的形式自由地关联，而读者自身也可以（或者说不得不）暂停了理性思考，在梦境一般的荒诞世界中一次又一次跳跃。而“翻译20遍”作为短视频，观看所需要的时间成本比正常的荒诞作品更低，仅仅抽出四五分钟（如果倍速播放则可以再减半）就可以体验类似的抽离和跳跃，这对于普遍学业繁重的青少年来说无疑更具吸引力。
“精神鸦片”，是的，你会这样说。但又有什么不好的呢？令人印象深刻的是，一位观众在评论区发布了自己的心路历程，讲述“翻译20遍”视频如何成为他（她）抗击抑郁症过程中最大的助力，作为深受学业压力、人际关系和心理问题困扰的高中生，在漫长的治疗与调整的过程中，遥远不可预知的正反馈如同远水救不了近火，而“翻译20遍”视频提供的易得的乐趣往往能在他（她）情绪最为低落时作为一剂“强心针”，成为治疗的有力补充与支持。
而且荒诞者并非只是为着荒诞而荒诞的。上世纪五六十年代的西方，伴随着世界大战的结束，在废墟上孑立的幸存者徒然试图恢复秩序，仅仅发现曾经信奉的人文主义与理性精神都已在历史的尘烟中化作模糊的虚影。在冷战的阴影下艰难前行，他们也许想要等待另一次文艺复兴的救赎，但上帝已经死了，戈多也不会来了，广大冰冷的宇宙没有中心，只是任凭每一道视线终结在黑暗的怀抱中。迷茫的一代由此而歌颂荒诞，用艺术再现了破碎的生活与文化的图景，通过浮夸与怪诞的表达绕过既定的规范，直抵美学的港湾，寻求心灵的庇护与疗救。而生活在今天的中国青少年，面临学业内卷愈发严重、教育资源分配不均、社会焦虑低龄化、个性发展受压抑的现状，正值探索与表达欲高涨的年龄，本来渴望沟通、知识、成长与爱，他们却往往受困于方寸课桌画地为牢。面对这种巨大的不对称性，“翻译20遍”中的荒诞言语恰好成为了他们的镜子，映照出同样荒诞的生活。视频结束，笑容消失，面对堆积如山的课业、难缠的校规校纪与互相竞争的同龄人，也许他们会问，“翻译20遍”和生活，到底哪个更荒诞一些？
此外，“翻译20遍”的另一特色是“金句”，往往指从反复翻译中产生、语言通顺而且可被解读出一定哲理的语句。这本来只是“猴子敲打字机”一类的概率现象（而且翻译软件生成这类句子的概率显然高于纯随机字符），但在完全由机器翻译形成的文本中诞生出“非人造的”警句箴言，这一现象本身也足够耐人寻味；翻译软件的生活化而非刻意文学化的语言风格也使得这些语句出现有别于“人造的”语录的风格特征。“人类双眼是透明的，我们涉过逆流的河水，向着光明的天空走去。”
二、技术与艺术：什么是创作，谁是我？
“翻译20遍”视频的创作过程，看似是一个顾名思义的问题，但实际上远不仅于此。首先是翻译次数和所用语种的选取，如果翻译遍数太少、选用的都是使用范围较广的语言，就常常出现与原文完全一致的现象，翻译软件大获全胜，而创作者无法得到预期的文本；反之，翻译次数太多、语种太生僻则可能导致大量句子成为完全无意义的乱码，甚至不能翻译回中文，也无法达成所需效果。而且这种反复翻译类似于物理中的混沌过程，具有不可预期、难以复现的特征，视频观众如果刨根问底、怀疑视频内容的真实性（文本是否为创作者人为编写），按照相同顺序用同一翻译引擎反复翻译同样文本同样次数，也仍然有可能得到不同的结果；而且为增加可读性和观赏效果，很多创作者会对文本进行个别字词的修改和调整，这些行为也会引发个别观众的质疑，认为创作者“欺骗”了他们。甚至有一位创作者不堪其扰，专门制作了一期视频讲解翻译和修改过程，以此和观众达成对于“翻译文本的人工修改”态度上的妥协。
然而在其他平台上的讨论呈现出截然相反的论调。在“怎么看待B站流行翻译20遍视频”的知乎提问下，回答者认为这种视频仅仅是把现成的文本材料用翻译软件来回翻译，没有进行任何创造性活动，属于完全的“低创”，甚至认为这些视频作品不应该享有原创标签和版权保护。这两种观点的割裂其实落脚在一个问题上：技术本身是否有创作的能力？
我窃以为技术是有创作的能力的。机器纺织的布料与人手织出的同样可以御寒，电脑计算的数据比人脑算出的更加可靠，那么为什么AI创作的诗歌、图画就不是诗歌、图画？为什么用软件反复翻译产生的作品就不是艺术？在观看者眼里，它们都是同等的。但“翻译20遍”的创作与人的创作又有着无法忽视的不同：作为人的我们想方设法捕捉稍纵即逝的灵感，绞尽脑汁把词句安置在适合的位置中；作为翻译软件却要经过人为的设障，把本来基本准确的翻译结果不厌其烦地添加上一层层扭曲的滤镜，最终形成滑稽可笑的破碎语句供人取乐。本是为超越和延拓人的能力而创造的技术又被我们拉低到低于常人的水平，好像绕了一圈回到原点，做功为零。
从《轩辕剑》系列游戏中无法被真正封存的“机关术”到赛博朋克之城血肉与机械融合的不谐交响，后现代视域下“技术”从未缺席人们的每一场焦虑和惶惑。机器越来越像人，从身体到智力再到创造力，属于人类的堡垒被机器逐一攻破。人却愈发像机器，“螺丝钉”的比喻已经不足以安抚个体置身群体中的迷茫和无助，现在我们所面对的是“原子化”，人与人之间的联系不断被削弱，算法编织的茧房舒适而安宁；人的全面发展的渴望不断让位于现实的竞争压力，我们在思维与能力上都被驱动着涌向流水线的模具，成为“单向度的人”。
这时候，人能做什么呢，也许只有自不量力地试图嘲笑机器吧？也许“翻译20遍”的创作本身，就是一群已经倦于与机器为伍的疲惫青年，给机器穿上了小丑的衣服，尽情地嘲笑它，用一种如同阿Q精神的方式提醒自己：我不是机器，我是人。也许他们可以接受与机关人谈笑风生，可以接受把自己的肢体换成机械义肢，但想必他们仍然不愿在灵魂上与机器混为一谈、接受主体性的泯灭与生活意义的全然否定。我是人，这是何等卑微，又何等高尚的认定啊！
三、补完计划：愿我们在塔中重聚
“我们放弃了评判辩论，代之以情感团结”，这是齐格蒙特∙鲍曼关于“怀旧”这一情绪的描述。我们总认为过去是好的，因为在那里一切的不确定性都已经被我们亲自蹚过，只留下安定的已知世界；而一切的恐惧、畏葸不前都来源于我们的视线无法穿透某处未知的迷雾。“他人即地狱”也许就是这种恐惧心理的极端表达，因为我们隔着彼此身体和意识的边界，我们无法真正相互理解，因此自己之外的每一双眼睛都是不可长久凝视的深渊。“把人变成机器”的工具理性的兴起看似为我们提供了一个一致的思考范式，通过削平个性、走向同质化来抹平个体之间的差异，但深陷其中的人们也仅仅是被封装进了一个个模式化的胶囊，在孤独的人群中，彼此的触碰反而更加遥不可及。
我们真的毫无办法了吗？也许并不是。荣格在《原型与集体无意识》中提出，在个体的意识、个体的潜意识之下还有更深层的称作集体无意识的心理领域。区别于个体的潜意识，它并不是由个人经验的积累产生，而是直接源自整个人类物种共有的、依靠基因传递的本能。就像世界上所有的岛屿都在海洋深处与整块大陆相连，个体彼此独立的精神深处也固有这种深层的连通。这种心理本能必然是非理性的，是先于人所学习的一切而存在的；如果要回到这种联结之中，再多的辩论也无济于事，只有艺术能做到。
回到“翻译20遍”本身，其创作方式与目的决定了它必然具有非理性的特征。我们不必为“非理性”这样的字眼而惶恐，更不必如同遇到瘟疫一般避之不及，因为非理性的精神力量是人本身的一部分，更进一步地，是按照设定程序运行的机器所永远无法拥有的，只属于真正的人的一部分。赫伊津哈在《游戏的人》的序言中说，我们的物种把自己和其他动物区分开来，我们曾经在那个遥远的乐观年代自诩“理性之人”，但我们的理性并不像我们所想的那般无所不能；现在我要把这个物种定义为“游戏的人”，这会更贴切些。沿着这个定义一路延伸到今天，我们的物种要把自己和机器区分开来，从游戏的精神中寻找一些线索，我们实际上在呼唤的就是非理性精神的回归。游戏的人，愿意编纂并遵守一套规则，克服本不必要的障碍，在没有现实收益的活动中乐此不疲。“他们疯了”，也许吧，因为他们终于不是，而且永远不再是机器了。赫伊津哈所说的“游戏精神”也正是艺术家的精神，“游戏的人”便是开掘了自身非理性的精神力量的鲜活的人。
再论何为“创作”的问题，如果把人的生活视作一件宏大的艺术品，那么阅读或欣赏本身可以是创作。“翻译20遍”视频的艺术性并不发生在使用翻译软件反复翻译的过程中，而是发生在观看的过程中。如果把有意义的文本翻译得支离破碎是一种意义的解构，那么从无意义的文本中获取愉悦与美感就是意义的新生。当背负着沉重的迷茫与憧憬的孩子暂时停下手中的笔，作为游戏的人观看一个由反复翻译产生的搞笑视频时，他就是自己生活的艺术家。身心快速发展的青少年终不会被禁锢在种种规训所划定的逼仄的道路中，即使在最繁重的学业压力下，也仍然有人在热烈地生活。“怀念高三”，这种说法看起来很奇怪，但事实上他们所怀念的不是囚笼一样的生活，而是试卷上的简笔画、草稿纸折的纸飞机、老师和同学的有趣绰号，是一次次肯定自己独一无二的创造迸发，是以炽热的艺术完成的对冰冷世界的突围。高塔直破苍穹，这一次即使是所谓的“上帝”也无法再阻止人类的抵达，因为艺术本身就是永恒的语言，跨越边界，填平鸿沟，与所有人同在。
四、永不落幕：生活还将继续
像任何流行文化一样，“翻译20遍”也迎来了不可避免的黄昏。这个诞生于青少年群体中、一开始就带着属于他们的浓厚底色的文化模因终而随着创作者们年龄的增长而一同走向衰老，目前最早一批创作者多数已经停止更新，平台流量有限、审美疲劳等因素也使得“翻译20遍”视频播放量不复往昔。但也不必遗憾叹惋，没有人能永葆青春，但永远有人正值青春；一种流行文化必有兴衰，但在鲜活的青年中必定会有新的文化层出不穷。
从去处来，我们再度审视“荒诞”本身，其真正的价值也不在于解构，而在于重建。假若只停留在解构的嘲弄一切与玩世不恭，荒诞便也被消解成了庸俗的狂欢，湮没了独立思考与人格，永陷于无深度的连绵泥淖中化与之同。“荒诞主义者在根本上不是虚无主义者，而是存在主义者”，荒诞之美在意义的重建中归来，用艺术的荒诞抵抗生活的荒诞，以精神的殿堂守望人生的旷野，站在解构的废墟之上，放声大笑宣告自己重获新生。
跨过“翻译20遍”这座里程碑，生活的艺术家们也将走上高考考场，向各自的目的地散去；但我们可以乐观地相信，这种不惟事功的精神能够伴随他们远离庸俗与市侩的泥淖，即使身处沟渠也不忘仰望星空。在现代性的洪流裹挟之下，我们常常感到迷失自我，在异化之下逐渐与机器混同；时代的病症需要时代的疗救，但对个人也并非无解，我们仍然可以常怀游戏精神，用开放的而非保守的心态、创造的而非固着的眼光对待自己的生活。互联网时代我们拥有更多的接收渠道与表达途径，同声相应，同气相求，同一频率的理想终将在大气层相逢。
“赐予我憧憬与回望，赐予我代码与诗歌”，我们需要相信人不仅仅是量化的各项功能的总和，鲜活的灵魂永远与机器不同，艺术能超越冰冷的钢铁穹顶。“世人皆知有用之用，而不知无用之用也。”也许正是“无用之用”，让千年之下铁马冰河以外的文明絮语有了不灭的余温。因为我们生而为人。
    ---
    """,
)


Admin (to chat_manager):



    Analyze the following text:


“翻译20遍”视频折射的青少年群体中后现代主体性困境与突围尝试

引言
家喻户晓的日本漫画《火影忍者》中有一个著名的设定，忍者使用一种叫做“查克拉”的能量，连接精神与身体，从而施放各种神奇效果的“忍术”。这一设定支撑了漫画前期到中期故事中所有的战斗情节乃至世界观，但在剧情后期，回溯上古时期“查克拉”的起源，它原本的用途是连接每个人的精神，使得人们越过个体感知与思维的界限，达成相互理解，消除隔阂与纷争，是为“忍道”；但“堕落的”人们不再用查克拉连接彼此的精神，而是连接自己的精神与自己的身体，创造出实质上与“忍道”相对立的“忍术”。形而上的“道”与形而下的“术”在这里直接对立，划开的是联结与孤立的分野。而原本为连接不同语言文化而生的翻译技术，也摇身一变成为搞笑视频的主角，云层之上居住的“上帝”是否也在嘲笑人类自己拆毁了高塔、折断了桥梁？
2020到2021年期间，在短视频平台哔哩哔哩（简称B站）上涌现了一批“xx（翻译软件）翻译xx（数字）遍xxxx（原文本名）”的视频，一般会打上“搞笑”标签，内容为把某一文本多次机翻再翻译回中文之后与原文逐句对照并打印在PPT中，使用AI配音朗诵，并配有背景音乐。而“翻译20遍”这个数字由于被选用较多，形成了流量聚集效应，成为了一个专属的分区标签。作为一个网络模因，“翻译20遍”的传播范围没有超出B站这一平台，但在站内获得了可观的播放量（单个视频播放量可超200万次）；该内容的制作者和观众（主要考虑评论区样本）也呈现出明显的年龄段聚集特征，集中在12-18岁的中学阶段。翻译的文本内容起初以中学课文为主，在2023年左右逐渐扩展到影视剧、动漫、广告、流行歌曲等等，乃至于B站其他的“鬼畜”（一类怪异、滑稽的剪辑视频的分区）作品也会被提取文字并翻译，再配上原视频形成新的“翻译20遍”作品。现今“翻译20遍”的黄金时期已然过去，以“鹰目大人”为代表的一众创作者由于学业压力等各种原因停止了投稿，新作的播放量也难以超过百万。但是作为一种凭借技术而生又置于艺术边界、在其他人群中鲜为人知却在青少年之间广为流传的文化模因，“翻译20遍”背后折射的青少年群体的迷茫与探索、狂欢与沉思足以引起我们更深远的关切。
一、荒诞与生活：从旧的迷

ChatResult(chat_id=None, chat_history=[{'content': '\n\n    Analyze the following text:\n\n\n“翻译20遍”视频折射的青少年群体中后现代主体性困境与突围尝试\n\n引言\n家喻户晓的日本漫画《火影忍者》中有一个著名的设定，忍者使用一种叫做“查克拉”的能量，连接精神与身体，从而施放各种神奇效果的“忍术”。这一设定支撑了漫画前期到中期故事中所有的战斗情节乃至世界观，但在剧情后期，回溯上古时期“查克拉”的起源，它原本的用途是连接每个人的精神，使得人们越过个体感知与思维的界限，达成相互理解，消除隔阂与纷争，是为“忍道”；但“堕落的”人们不再用查克拉连接彼此的精神，而是连接自己的精神与自己的身体，创造出实质上与“忍道”相对立的“忍术”。形而上的“道”与形而下的“术”在这里直接对立，划开的是联结与孤立的分野。而原本为连接不同语言文化而生的翻译技术，也摇身一变成为搞笑视频的主角，云层之上居住的“上帝”是否也在嘲笑人类自己拆毁了高塔、折断了桥梁？\n2020到2021年期间，在短视频平台哔哩哔哩（简称B站）上涌现了一批“xx（翻译软件）翻译xx（数字）遍xxxx（原文本名）”的视频，一般会打上“搞笑”标签，内容为把某一文本多次机翻再翻译回中文之后与原文逐句对照并打印在PPT中，使用AI配音朗诵，并配有背景音乐。而“翻译20遍”这个数字由于被选用较多，形成了流量聚集效应，成为了一个专属的分区标签。作为一个网络模因，“翻译20遍”的传播范围没有超出B站这一平台，但在站内获得了可观的播放量（单个视频播放量可超200万次）；该内容的制作者和观众（主要考虑评论区样本）也呈现出明显的年龄段聚集特征，集中在12-18岁的中学阶段。翻译的文本内容起初以中学课文为主，在2023年左右逐渐扩展到影视剧、动漫、广告、流行歌曲等等，乃至于B站其他的“鬼畜”（一类怪异、滑稽的剪辑视频的分区）作品也会被提取文字并翻译，再配上原视频形成新的“翻译20遍”作品。现今“翻译20遍”的黄金时期已然过去，以“鹰目大人”为代表的一众创作者由于学业压力等各种原因停止了投稿，新作的播放量也难以超过百万。但是作为一种凭借技术而生又置于艺术边界、在其他人群中鲜为人知却在青少年之间广为流传的文化模因，“翻译20遍”背后折射的青少年群体的迷茫